> Bản dịch tiếng Việt · nguồn: [sft_math.ipynb](../../documents/chapter4/sft_math.ipynb) · 《动手学大模型》(Lordog/dive-into-llms)


In [ ]:
import transformers
import torch
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
import matplotlib.pyplot as plt
from vllm import LLM, SamplingParams
import copy

### Thông tin môi trường thực thi cơ bản

In [ ]:
print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)
print("Datasets version:", datasets.__version__)
print("VLLM version:", transformers.__version__)

In [ ]:
print("Number of CUDA devices:", torch.cuda.device_count())
print("Current CUDA device:", torch.cuda.current_device())

### Tải bộ dữ liệu
Trong thí nghiệm này, chúng tôi sử dụng bộ dữ liệu huấn luyện suy luận toán học [DeepMath-103K](https://huggingface.co/datasets/zwhe99/DeepMath-103K). DeepMath-103K là một bộ dữ liệu toán học độ khó cao, gồm 103K mẫu; mỗi mẫu đi kèm nhãn độ khó cùng ba đường hướng suy luận (reasoning path) do DeepSeek-R1 sinh ra. Ngoài ra, một đặc điểm nổi bật của DeepMath-103K là thao tác loại bỏ trùng lặp (deduplication) hướng tới các bộ test toán học phổ biến hiện có, đảm bảo tập huấn luyện không chứa mẫu của các bộ test toán học chủ lưu.

In [ ]:
# Sử dụng api của datasets để nạp bộ dữ liệu
data = load_dataset("zwhe99/DeepMath-103K")

In [ ]:
data

### Tokenizer và chat_template
Tokenizer có vai trò chuyển văn bản ngôn ngữ tự nhiên dạng chuỗi thành đầu vào mà mô hình có thể chấp nhận (một danh sách id số nguyên). chat_template là thành phần đặc trưng của mô hình lớn, định nghĩa cách tổ chức một request thành dạng hội thoại.

In [ ]:
# Nạp Tokenizer
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Math-1.5B")
print(qwen_tokenizer.chat_template)

In [ ]:
# Phương thức apply_chat_template chuyển một lịch sử hội thoại thành định dạng đầu vào có token đặc biệt
# Ở đây tokenize=False nghĩa là không phân từ, add_generation_prompt=False nghĩa là không thêm prompt sinh
# Có thể quan sát từ ví dụ sinh ra cách nội dung đầu vào thực tế của người dùng được tổ chức
text = "hello world"
print(qwen_tokenizer.apply_chat_template([{"role": "user", "content": text}, {"role": "assistant", "content": "hi!"}], tokenize=False, add_generation_prompt=False))

### Tiền xử lý bộ dữ liệu
Chúng tôi thực hiện chủ yếu hai bước tiền xử lý dữ liệu sau:
1. Lọc dữ liệu, chỉ giữ lại các bài toán có đáp án là số thuần
2. Lọc theo độ dài, chỉ giữ lại các bài toán có độ dài nhỏ hơn 4096

In [ ]:
# Lọc dữ liệu: chỉ giữ các bài có đáp án là số thuần
data_with_number_answer = data["train"].filter(
    lambda x: x["final_answer"].isdigit()
)
print(len(data_with_number_answer))

In [ ]:
# Tiền xử lý dữ liệu: mỗi bài toán lấy output đầu tiên của DeepSeek-R1 làm đáp án chuẩn, xử lý thành định dạng gồm độ dài (tiện cho lọc về sau), input_ids và labels (phục vụ huấn luyện)
def tokenize(example):
    question = example["question"]
    r1_solution = example["r1_solution_1"]
    message = [{"role": "user", "content": question}, {"role": "assistant", "content": f"<think>\n{r1_solution}\n"}]
    result = qwen_tokenizer.apply_chat_template(message, add_generation_prompt=False, tokenize=True, return_dict=True)
    result["length"] = len(result["input_ids"])
    result["labels"] = result["input_ids"].copy()
    return result

train = data_with_number_answer.map(tokenize, batched=False, num_proc=16)

In [ ]:
# Ví dụ dữ liệu: câu trả lời của R1 chứa các đặc trưng suy luận cấp cao như tự phản tư, chuyển hướng ý tưởng, kiểm tra lại phép tính
# Đây là mẫu thực sự được truyền vào mô hình để huấn luyện
print(qwen_tokenizer.decode(train[0]["input_ids"], skip_special_tokens=False))

#### Lọc theo độ dài
Để thuận tiện cho huấn luyện, chúng tôi lấy độ dài tối đa 4096 của Qwen/Qwen2.5-Math-1.5B làm ngưỡng và chỉ giữ lại các mẫu có độ dài nhỏ hơn độ dài tối đa của mô hình

In [ ]:
# Trực quan hóa phân bố độ dài: độ dài trung bình câu trả lời của R1 vào khoảng 5000
lengths = train["length"]
plt.hist(lengths, bins=100)
plt.xlabel("Length")
plt.ylabel("Count")
plt.title("Length Distribution")
plt.show()

In [ ]:
# Lọc dữ liệu: chỉ giữ các mẫu có độ dài nhỏ hơn 4096
train = train.filter(lambda x: x["length"] < 4096, num_proc=16)
print(len(train))

### Huấn luyện mô hình
Để thuận tiện cho thí nghiệm, chúng tôi dùng mô hình Qwen/Qwen2.5-Math-1.5B làm nền của thí nghiệm. Qwen/Qwen2.5-Math-1.5B là một mô hình nền (base model) chưa qua fine-tuning theo chỉ dẫn, nên không thể đưa ra đầu vào chỉ dẫn một cách chuẩn mực mà chỉ có thể sinh theo kiểu ICL thông thường. Chúng tôi sẽ nạp mô hình này và fine-tune với độ chính xác BF16. Thí nghiệm sử dụng một GPU A100 80GB.

In [ ]:
# Nạp mô hình
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Math-1.5B", torch_dtype=torch.bfloat16, device_map="auto")
model

In [ ]:
# Thiết lập tham số huấn luyện
training_args = TrainingArguments(
    # batch size & epochs
    per_device_train_batch_size=4, # Khi bộ nhớ GPU không đủ, hãy đặt thành 1
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    # hyperparameters
    learning_rate=1e-6,
    lr_scheduler_type="cosine",
    # monitoring
    output_dir="./checkpoints",
    logging_dir="./checkpoints/logs",
    report_to="tensorboard",
    eval_strategy="no",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    # efficiency
    bf16=True,
    group_by_length=True,
    torch_compile=True,
    gradient_checkpointing=False, # Khi bộ nhớ GPU không đủ, hãy đặt thành True
    # reproducibility
    seed=42,
    data_seed=42
)

# Thiết lập DataCollator mặc định cho tác vụ seq2seq
collator = DataCollatorForSeq2Seq(
    qwen_tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

In [ ]:
# Huấn luyện bằng Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    processing_class=qwen_tokenizer,
    data_collator=collator,
)
trainer.train()

In [ ]:
# Sau khi huấn luyện xong, lưu mô hình cuối cùng
trainer.save_model("./checkpoints/final_model")

### Kiểm thử
Sau khi huấn luyện xong, chúng ta cần kiểm thử hiệu năng của mô hình thu được. Chúng tôi sử dụng bộ test math500 được dùng rộng rãi. math500 là tập con của bộ test [MATH](https://datasets-benchmarks-proceedings.neurips.cc/paper_files/paper/2021/file/be83ab3ecd0db773eb2dc1b0a17836a1-Paper-round2.pdf), gồm 500 bài toán toán học cấp độ thi đấu trung học phổ thông. Mỗi bài toán đều được gán nhãn độ khó tương ứng.

In [ ]:
# Nạp dữ liệu
math500 = load_dataset("zwhe99/MATH", split="math500")

In [ ]:
# Quan sát định dạng dữ liệu
math500[0]

In [ ]:
## Tiền xử lý: xử lý bài toán thành định dạng hội thoại phù hợp đầu vào của mô hình 
data = []
for line in math500:
    line = copy.deepcopy(line)
    line["messages"] = [{"role": "user", "content": line["problem"]}]
    data.append(line)
print(len(data))

#### Mô hình sinh kết quả
Chúng tôi sử dụng vllm làm backend suy luận. Đường dẫn `base_model` cần được điền đường dẫn thực tế của mô hình muốn kiểm thử.

In [ ]:
base_model = "./checkpoints/final_model"
llm = LLM(
    model=base_model,
    tensor_parallel_size=1,
    max_model_len=4096
)
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [ ]:
# Chuyển đầu vào thành định dạng Prompt: VLLM suy luận trực tiếp cần đầu vào đã qua phương thức apply_chat_template
prompts = [
    tokenizer.apply_chat_template(
        conversation=line["messages"],
        tokenize=False,
        add_generation_prompt=True,
    )
    for line in data
]

# Đồng nhất với định dạng huấn luyện: thêm marker <think> để kích hoạt mô hình suy nghĩ
prompts = [prompt + "<think>" for prompt in prompts]  # Add newline at the end of each prompt
# Thiết lập tham số sinh: temperature=0 nghĩa là giải mã tham lam (greedy), max_tokens=3700 để phù hợp độ dài tối đa của mô hình
sampling_params = SamplingParams(temperature=0, max_tokens=3700)
# Tiến hành sinh
outputs = llm.generate(
    prompts,
    sampling_params,
)

### Đánh giá toán học
Ở phần này, chúng tôi sử dụng phương pháp khớp mẫu (rule matching) dựa trên luật để xác định đáp án toán học đúng hay sai. Trước hết cần rút trích giá trị đáp án cuối cùng từ câu trả lời được sinh ra. Theo thiết lập lúc huấn luyện, đáp án sinh ra thường nằm trong nhãn `\boxed{}`, nên chúng ta có thể dùng biểu thức chính quy (regular expression) để rút trích đáp án từ kết quả sinh, sau đó so sánh đáp án đó với đáp án đúng

In [ ]:
# Cài đặt các dependency cần thiết
!pip install antlr4-python3-runtime==4.11.0

In [ ]:
from openmathinst_utils import math_equal, extract_answer

In [ ]:
# Xử lý output đã sinh, rút trích đáp án ra ngoài
for line, output in zip(data, outputs):
    line["response"] = output.outputs[0].text
    line["extracted_answer"] = extract_answer(output.outputs[0].text)

In [ ]:
# Dùng phương thức math_equal để kiểm tra đáp án đã rút trích đúng hay sai
for line in data:
    line["correct"] = math_equal(line["extracted_answer"], line["expected_answer"])
print("Accuracy:", sum(line["correct"] for line in data) / len(data))

### Case Study
Ở phần này, chúng ta xem ngẫu nhiên nội dung do mô hình sinh ra để quan sát đặc điểm sinh của mô hình. Qua việc quan sát mẫu ngẫu nhiên, có thể nhận thấy:
1. Mô hình sau fine-tuning đã có khả năng tuân thủ chỉ dẫn (instruction following) ở mức cơ bản
2. Mô hình xuất hiện khuynh hướng phản tư (reflection) rõ rệt với nhiều từ "confirm", và có xu hướng dùng mã (code) để kiểm tra đáp án

In [ ]:
import random

In [ ]:
item = random.choice(data)
print(item["problem"])
print("="*50)
print(item["response"])